In [ ]:
%pip install duckdb

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, RepeatVector, TimeDistributed
import duckdb as db

In [3]:

modelDat = db.sql(f'''
                    
    select 
    * 
    from read_parquet('C:/MLAR_DATA/T0001/MLAR_RAW_X5.parquet') 
    
    ''').to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
features = ['AUDIT_ACTION','AUDIT_TIMESTAMP','AUDIT_TYPE','TABLE_NAME', 'TABLE_KEY','AUDIT_COUNTER','TRANS_ORDER']

In [5]:
modelDat = modelDat[features]

In [6]:
modelDat['namekey'] = modelDat['TABLE_NAME'] + '-' + modelDat['TABLE_KEY'].astype(str)
modelDat


,AUDIT_ACTION,AUDIT_TIMESTAMP,AUDIT_TYPE,TABLE_NAME,TABLE_KEY,AUDIT_COUNTER,TRANS_ORDER,namekey
0,Insert,2023-11-01 12:40:10,LogSample,SAMPLE,45327,3,3,SAMPLE-45327
1,Update,2023-11-01 12:40:10,ReceiveSample,TEST,80011,1,1,TEST-80011
2,Insert,2023-11-06 21:40:15,EditTests,TEST,80312,3,3,TEST-80312
3,Update,2023-11-06 21:40:28,ModifySample,SAMPLE,45088,2,2,SAMPLE-45088
4,Insert,2023-11-01 12:41:18,LogSample,SAMPLE,45328,1,1,SAMPLE-45328
...,...,...,...,...,...,...,...,...
571302,Insert,2023-11-01 12:40:10,LogSample,RESULT,345294,1,1,RESULT-345294
571303,Insert,2023-11-01 12:40:10,LogSample,TEST,80011,2,2,TEST-80011
571304,Update,2023-11-01 12:40:10,ReceiveSample,SAMPLE,45327,2,2,SAMPLE-45327
571305,Update,2022-03-15 19:22:13,Authorize,RESULT,200818,94,94,RESULT-200818


In [7]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load your data
# df = pd.read_csv('your_data.csv')

# Assuming df is your DataFrame
# Convert audit_timestamp to datetime
modelDat['AUDIT_TIMESTAMP'] = pd.to_datetime(modelDat['AUDIT_TIMESTAMP'])

# Extract time features
modelDat['hour'] = modelDat['AUDIT_TIMESTAMP'].dt.hour
modelDat['day_of_week'] = modelDat['AUDIT_TIMESTAMP'].dt.weekday

# Normalize time features
scaler = MinMaxScaler()
modelDat[['hour']] = scaler.fit_transform(modelDat[['hour']])

# One-hot encode categorical variables
encoder = OneHotEncoder(sparse_output=False)

encoded_cats = encoder.fit_transform(modelDat[['AUDIT_TYPE', 'AUDIT_ACTION']])
encoded_cats_df = pd.DataFrame(encoded_cats)

# Concatenate encoded categorical data back to the main DataFrame
modelDat = pd.concat([modelDat, encoded_cats_df], axis=1)

modelDat_sorted = modelDat.sort_values(by=['namekey', 'AUDIT_TIMESTAMP'])


modelDat.drop(['TABLE_NAME','TABLE_KEY','AUDIT_TYPE', 'AUDIT_ACTION'],axis=1)


# Prepare sequences: here you need to group by 'key' and possibly sort by 'audit_timestamp'
grouped = modelDat.groupby('namekey')

# Assuming each group is now a separate sequence, you'd pad sequences here
# Pad your sequences using pad_sequences from Keras, if using LSTMs

# Now, you would split this data into features and labels as needed, reshape it, and feed into your LSTM model


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Masking

# Assuming you've chosen a number of features per step and fixed the sequence length (e.g., 90)
sequence_length = 273  # Max sequence length after padding
feature_columns =[col for col in modelDat.columns if col not in ['namekey', 'AUDIT_TIMESTAMP','TABLE_NAME','TABLE_KEY','AUDIT_TYPE', 'AUDIT_ACTION']]
number_of_features = len(feature_columns) # Adjust based on actual number of features after preprocessing

model = Sequential([
    # Masking layer to ignore padded values
    Masking(mask_value=0., input_shape=(sequence_length, number_of_features)),
    # Encoder
    LSTM(128, activation='relu', return_sequences=True),
    LSTM(64, activation='relu', return_sequences=False),
    RepeatVector(sequence_length),  # Prepares the fixed-length vector for decoding
    # Decoder
    LSTM(64, activation='relu', return_sequences=True),
    LSTM(128, activation='relu', return_sequences=True),
    TimeDistributed(Dense(number_of_features))  # Output layer to reconstruct the original input
])

model.compile(optimizer='adam', loss='mse')
model.summary()


c:\Users\dan.larson\Documents\MLAR\.venv\Lib\site-packages\keras\src\layers\core\masking.py:47: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 273, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 273, 128)       │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 273, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 273, 64)        │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 273, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 273, 32)        │         4,128 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 267,808 (1.02 MB)

 Trainable params: 267,808 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
grouped = modelDat.groupby('namekey')

In [9]:
feature_columns = [col for col in modelDat.columns if col not in ['namekey', 'AUDIT_TIMESTAMP','TABLE_NAME','TABLE_KEY','AUDIT_TYPE', 'AUDIT_ACTION']]
sequences = [group[feature_columns].values for namekey, group in grouped]

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad sequences
padded_sequences = pad_sequences(sequences, maxlen=273, padding='post', dtype='float32')

In [11]:
from sklearn.model_selection import train_test_split

# Assuming 'padded_sequences' is your final dataset ready for training
X_train, X_val = train_test_split(padded_sequences, test_size=0.2, random_state=42)

In [12]:
history = model.fit(X_train, X_train, 
                    epochs=50, 
                    batch_size=32, 
                    validation_data=(X_val, X_val))

Epoch 1/50
4576/4576 ━━━━━━━━━━━━━━━━━━━━ 2714s 586ms/step - loss: nan - val_loss: nan
Epoch 2/50
1197/4576 ━━━━━━━━━━━━━━━━━━━━ 31:52 566ms/step - loss: nan

: 

In [ ]:
# Calculate reconstruction loss on validation data
predicted = model.predict(X_val)
reconstruction_error = np.mean(np.power(X_val - predicted, 2), axis=1)

# You can plot or statistically examine reconstruction error here

In [ ]:
threshold = np.quantile(reconstruction_error, 0.95)  # Setting a threshold as the 95th percentile of error
outliers = reconstruction_error > threshold

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.legend()
plt.show()